# Week 4 Exercise – Code Generator (winniekariuki)

Use LLMs to **convert Python to C++**, **add docstrings and comments**, and **write unit tests**. Uses **OpenAI** (GPT-4o-mini) and **Llama 3.2** (Ollama) with a **Gradio UI**.

In [ ]:
# imports
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# environment & API clients (same pattern as Week 2 & 3)
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openai_client = OpenAI() if (api_key and api_key.startswith("sk-") and len(api_key) > 10) else None
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

if openai_client:
    print("OpenAI client ready (GPT).")
else:
    print("OPENAI_API_KEY not set — add to .env to use GPT.")
print("Ollama client ready (Llama). Ensure Ollama is running: ollama run llama3.2")

## Code generator actions

Three actions: **Convert to C++**, **Add docstrings/comments**, and **Write unit tests**.

In [ ]:
# Actions and prompts
ACTIONS = ["Convert to C++", "Add docstrings & comments", "Write unit tests"]

SYSTEM_PROMPTS = {
    "Convert to C++": """You are an assistant that reimplements Python code in high-performance C++.
Respond only with C++ code. Use comments sparingly. Do not provide explanations outside the code.
The C++ must produce identical output. Include all necessary headers (e.g. iostream, iomanip).""",
    "Add docstrings & comments": """You are an assistant that adds PEP-257 docstrings and succinct inline comments to Python code.
Respond only with valid Python code. Preserve the original logic; add docstrings to modules, classes, and functions.""",
    "Write unit tests": """You are an assistant that creates pytest unit tests for Python code.
Respond only with valid Python code. Use pytest; cover main branches and edge cases; keep tests clear and focused.""",
}

USER_PROMPTS = {
    "Convert to C++": "Rewrite this Python code in C++ with a fast implementation that produces identical output. Respond only with C++ code; no explanations. Pay attention to number types (no int overflow).",
    "Add docstrings & comments": "Add appropriate docstrings and comments to this Python code. Respond only with the updated Python code.",
    "Write unit tests": "Create pytest unit tests for this Python code. Respond only with the test Python code.",
}

In [ ]:
def strip_code_block(raw: str) -> str:
    """Remove markdown code fences if present."""
    raw = (raw or "").strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```\w*\n", "", raw)
        raw = re.sub(r"\n```\s*$", "", raw)
    return raw


def generate_code(python_code: str, action: str, model_label: str) -> str:
    """Generate code using OpenAI or Ollama. Returns raw code string."""
    if not python_code or not python_code.strip():
        return "Please provide Python code."

    action = action or ACTIONS[0]
    system = SYSTEM_PROMPTS.get(action, SYSTEM_PROMPTS["Convert to C++"])
    user_prefix = USER_PROMPTS.get(action, USER_PROMPTS["Convert to C++"])
    user = f"{user_prefix}\n\n---\n\n{python_code.strip()}"

    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]

    try:
        if model_label == "GPT-4o-mini (OpenAI)":
            if not openai_client:
                return "OpenAI client not configured. Set OPENAI_API_KEY in .env."
            r = openai_client.chat.completions.create(
                model=MODEL_GPT,
                messages=messages,
                temperature=0.3,
            )
        elif model_label == "Llama 3.2 (Ollama)":
            r = ollama_client.chat.completions.create(
                model=MODEL_LLAMA,
                messages=messages,
                temperature=0.3,
            )
        else:
            return "Unknown model. Choose GPT-4o-mini (OpenAI) or Llama 3.2 (Ollama)."

        raw = (r.choices[0].message.content or "").strip()
        return strip_code_block(raw)
    except Exception as e:
        return f"Error: {e}"

## Gradio UI

Paste Python code, pick an action (Convert to C++, Add docstrings/comments, Write unit tests), choose a model, and generate.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Code Generator – Week 4 (winniekariuki)") as demo:
    gr.Markdown(
        "# Code Generator\n"
        "Paste Python code below, pick an **action** (Convert to C++, Add docstrings & comments, Write unit tests), choose **OpenAI** or **Llama**, and generate."
    )
    with gr.Row():
        action = gr.Dropdown(ACTIONS, value=ACTIONS[0], label="Action")
        model_choice = gr.Radio(
            ["GPT-4o-mini (OpenAI)", "Llama 3.2 (Ollama)"],
            label="Model",
            value="GPT-4o-mini (OpenAI)",
        )
    python_code = gr.Textbox(
        label="Python code",
        placeholder="Paste your Python code here...",
        lines=12,
    )
    btn = gr.Button("Generate", variant="primary")
    output = gr.Textbox(label="Generated output", lines=14)

    btn.click(
        fn=generate_code,
        inputs=[python_code, action, model_choice],
        outputs=output,
    )

demo.launch(inbrowser=True)